# Understanding Model Config Files

> In the previous sections, we have been building everything from scratch: writing the Tokenizer, building the Embedding, stacking Transformer Blocks -- all structural parameters were hardcoded in Python code, so changing a hidden dimension meant rewriting the code. Real-world large models are not organized and distributed this way.
>
> This section opens the SmolLM2-135M repository to see what files a modern LLM is actually made of, what each file controls, and how to use these files to load and run a model. We switch from "writing code to define a model" to "reading configs to describe a model."


SmolLM2-135M's `config.json` contains three numbers that directly determine Attention shapes. Each head has dimension `576 / 9 = 64`. Q uses nine heads and therefore has width 576; K and V use three heads each and therefore have width 192. The K projection weight must have shape `[192, 576]`.

If code reads only `hidden_size=576` and assumes nine K heads, it creates `[576, 576]`, which cannot load a checkpoint tensor shaped `[192, 576]`. No numerical data is missing; the reconstructed architecture is wrong.

The parameters that describe layer count, width, and component variants are the **model configuration**. Loading first creates parameters from this configuration and then places checkpoint values into matching tensors. We begin with the two K projections and observe how a configuration error becomes a shape conflict.


## 0. Model Structure and Model Weights

Retain only the K projection. Both Linear layers receive 576-dimensional input; one outputs 192 values and the other 576. Loading the same K weight into each exposes the structural mismatch directly.


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

saved_k_weight = torch.randn(192, 576)
correct_k_proj = nn.Linear(576, 192, bias=False)
wrong_k_proj = nn.Linear(576, 576, bias=False)

print("K matrix in the weight file:", list(saved_k_weight.shape))
print("Parameter created by the correct config: ", list(correct_k_proj.weight.shape))
print("Parameter created by the wrong config:   ", list(wrong_k_proj.weight.shape))
print("\nKey observation: only shapes created from the correct configuration align with every stored weight.")
assert saved_k_weight.shape == correct_k_proj.weight.shape
assert saved_k_weight.shape != wrong_k_proj.weight.shape


## 1. Implementing Attention from a Configuration

Before examining every field, implement an Attention class that reads dimensions from `config`, then run the same input through several configurations.

`hidden_size` controls input/output width, `num_attention_heads` controls Q heads, and `num_key_value_heads` controls K/V heads. Changing these values creates a different Attention structure.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class MultiHeadAttention(nn.Module):
    """Create Multi-Head Attention projection matrices from a config.

    This example uses only three config fields:
      hidden_size          input and output width d
      num_attention_heads  number of Q heads h
      num_key_value_heads  number of K/V heads kv (standard MHA when equal to h)
    """
    def __init__(self, config):
        super().__init__()
        d = config['hidden_size']
        h = config['num_attention_heads']
        kv = config['num_key_value_heads']
        hd = d // h                       # dimension of each head
        self.h, self.kv, self.groups, self.head_dim = h, kv, h // kv, hd
        # The widths of all four projections follow completely from these three numbers
        self.q_proj = nn.Linear(d, h * hd, bias=False)
        self.k_proj = nn.Linear(d, kv * hd, bias=False)
        self.v_proj = nn.Linear(d, kv * hd, bias=False)
        self.o_proj = nn.Linear(h * hd, d, bias=False)

    def forward(self, x):
        b, t, d = x.shape
        q = self.q_proj(x).view(b, t, self.h, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.kv, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv, self.head_dim).transpose(1, 2)
        # When there are fewer K/V heads, each K/V group serves several Q heads—the core of GQA
        k = k.repeat_interleave(self.groups, dim=1)
        v = v.repeat_interleave(self.groups, dim=1)
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = attn.transpose(1, 2).reshape(b, t, -1)
        return self.o_proj(out)

print("MultiHeadAttention is defined. In the next cell, run it with different configs.")


The same class becomes MHA with nine K/V heads, SmolLM2's GQA with three, or MQA with one. Run each on the same random input and compare K projection shape and parameter count.


In [ ]:
variants = {
    'MHA (kv=9)':  {'hidden_size': 576, 'num_attention_heads': 9, 'num_key_value_heads': 9},
    'GQA (kv=3)':  {'hidden_size': 576, 'num_attention_heads': 9, 'num_key_value_heads': 3},
    'MQA (kv=1)':  {'hidden_size': 576, 'num_attention_heads': 9, 'num_key_value_heads': 1},
}

x = torch.randn(2, 16, 576)  # feed the same input to all three models

results = {}
for name, cfg in variants.items():
    torch.manual_seed(42)               # fix the seed so differences come only from structure
    model = MultiHeadAttention(cfg)
    out = model(x)
    params = sum(p.numel() for p in model.parameters())
    results[name] = params
    print(f"{name}: K weight {list(model.k_proj.weight.shape)}, "
          f"parameters {params:,}, output {list(out.shape)}")

print("\nKey observation: input and output remain [2, 16, 576]; all differences are hidden in intermediate parameter shapes.")


In [ ]:
import matplotlib.pyplot as plt

names = list(results)
values = [results[n] / 1e6 for n in names]

plt.figure(figsize=(6, 4))
bars = plt.bar(names, values, color=['#5DADE2', '#F5B041', '#E74C3C'])
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f'{v:.2f}M', ha='center')
plt.ylabel('Attention parameters (M)')
plt.title('Same class, same input, different configs')
plt.tight_layout()
plt.show()

## 2. `config.json` and Model Structure

Open SmolLM2-135M's [`config.json`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/main/config.json). Rather than memorizing more than twenty fields, identify six values that directly change parameter shapes.

Follow the data flow: how large is the token Embedding, how wide is a token vector, how does Attention split heads, and how wide does the FFN expand? This order connects fields to computation.


In [ ]:
config = {
    "architectures": ["LlamaForCausalLM"],
    "hidden_size": 576,
    "intermediate_size": 1536,
    "num_attention_heads": 9,
    "num_key_value_heads": 3,
    "num_hidden_layers": 30,
    "vocab_size": 49152,
    "max_position_embeddings": 8192,
    "hidden_act": "silu",
    "rms_norm_eps": 1e-05,
    "rope_theta": 100000,
    "tie_word_embeddings": True,
    "attention_bias": False,
}

V, D, L, H, KV, FF = (config[k] for k in (
    "vocab_size", "hidden_size", "num_hidden_layers",
    "num_attention_heads", "num_key_value_heads", "intermediate_size"))
head_dim = D // H

assert D % H == 0, "hidden_size must be divisible by the number of Attention heads"
assert H % KV == 0, "The number of Q heads must be divisible by the number of KV heads"

print(f"Embedding matrix: [{V}, {D}]")
print(f"Each Attention head: {D} / {H} = {head_dim} dimensions")
print(f"Every {H // KV} Q heads share one K/V group, for {KV} groups total")
print(f"FFN in each Block: {D} -> {FF} -> {D}")
print(f"The model stacks {L} Blocks")


### 2.1 The Parameter Skeleton

Use the configuration to create one Block's parameter skeleton: two RMSNorms, four Attention projections, and three SwiGLU projections. The simplified forward pass checks shapes and counts but omits the causal mask and RoPE, so it is not a complete SmolLM2 reproduction.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    """Create a Transformer Block parameter skeleton from the config for shape verification."""
    def __init__(self, config):
        super().__init__()
        d = config['hidden_size']
        ff = config['intermediate_size']
        h = config['num_attention_heads']
        kv = config['num_key_value_heads']
        hd = d // h
        bias = config['attention_bias']

        self.attn_norm = nn.RMSNorm(d, eps=config['rms_norm_eps'])
        self.ffn_norm  = nn.RMSNorm(d, eps=config['rms_norm_eps'])
        # Attention: four projections
        self.q_proj = nn.Linear(d, h * hd, bias=bias)
        self.k_proj = nn.Linear(d, kv * hd, bias=bias)
        self.v_proj = nn.Linear(d, kv * hd, bias=bias)
        self.o_proj = nn.Linear(h * hd, d, bias=bias)
        # FFN: three projections
        self.gate = nn.Linear(d, ff, bias=False)
        self.up   = nn.Linear(d, ff, bias=False)
        self.down = nn.Linear(ff, d, bias=False)

        self.n_heads = h
        self.n_kv_heads = kv
        self.head_dim = hd

    def forward(self, x):
        # Attention (simplified, no causal mask or RoPE)
        residual = x
        x = self.attn_norm(x)
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        # GQA broadcast
        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        x = residual + self.o_proj(out)
        # FFN
        residual = x
        x = self.ffn_norm(x)
        x = residual + self.down(F.silu(self.gate(x)) * self.up(x))
        return x

# Build one Block using SmolLM2's config
block = TransformerBlock(config)
print("=== Complete structure of a single TransformerBlock ===")
print(block)
block_params = sum(p.numel() for p in block.parameters())
print(f"\nParameters in this Block: {block_params:,}  ({block_params/1e6:.2f}M)")


## 3. Q Heads and K/V Heads

Ordinary MHA would pair nine Q heads with nine K and V heads. SmolLM2 instead sets `num_attention_heads=9` and `num_key_value_heads=3`.

**Grouped-Query Attention (GQA)** lets one K/V head serve several Q heads. With head dimension $576/9=64$, Q outputs $9\times64=576$ values, while K and V each output $3\times64=192$. We create the four projections to verify these shapes.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

W_q = nn.Linear(D, H * head_dim, bias=False)
W_k = nn.Linear(D, KV * head_dim, bias=False)
W_v = nn.Linear(D, KV * head_dim, bias=False)
W_o = nn.Linear(H * head_dim, D, bias=False)

x = torch.randn(2, 16, D)  # Simulated hidden states
q = W_q(x).view(2, 16, H, head_dim).transpose(1, 2)   # [2, 9, 16, 64]
k = W_k(x).view(2, 16, KV, head_dim).transpose(1, 2)  # [2, 3, 16, 64]
v = W_v(x).view(2, 16, KV, head_dim).transpose(1, 2)  # [2, 3, 16, 64]

print(f"Input: [2, 16, {D}]")
print(f"Q projection: {list(W_q.weight.shape)} -> Q shape: {list(q.shape)}")
print(f"K projection: {list(W_k.weight.shape)} -> K shape: {list(k.shape)}")
print(f"V projection: {list(W_v.weight.shape)} -> V shape: {list(v.shape)}")
print(f"O projection: {list(W_o.weight.shape)}")
print()
q_p = sum(p.numel() for p in W_q.parameters())
k_p = sum(p.numel() for p in W_k.parameters())
print(f"Q params: {q_p:,}   K params: {k_p:,}   K/Q = {k_p/q_p:.2f}")
print(f"If this were MHA (Q=K=V=9): K params would also be {q_p:,}")
saved_kv_params = 2 * (q_p - k_p) * L
print(f"GQA saves {saved_kv_params:,.0f} K+V parameters across {L} layers")

plt.figure(figsize=(6, 3.5))
plt.bar(["Q", "K", "V"], [q_p, k_p, k_p],
        color=["#4C78A8", "#F58518", "#F58518"])
plt.ylabel("Parameters")
plt.title("Projection parameters in one GQA block")
plt.tight_layout()
plt.show()


How do nine Q heads attend with three K/V heads? Logically repeat each K/V group three times so the head dimension aligns. The important fact is sharing rather than physical copying: `KV[0]` serves `Q[0:3]`, `KV[1]` serves `Q[3:6]`, and `KV[2]` serves `Q[6:9]`.


In [ ]:
groups = H // KV  # 3 Q heads per group
k_repeated = k.repeat_interleave(groups, dim=1)  # [2, 3, 16, 64] → [2, 9, 16, 64]
v_repeated = v.repeat_interleave(groups, dim=1)

# Now Q and K have the same number of heads, so we can compute Attention normally
scale = head_dim ** -0.5
attn_weights = (q @ k_repeated.transpose(-2, -1)) * scale  # [2, 9, 16, 16]

print("Original K shape:", list(k.shape), "-> after repetition:", list(k_repeated.shape))
print("Original V shape:", list(v.shape), "-> after repetition:", list(v_repeated.shape))
print(f"QK^T result:      {list(attn_weights.shape)}")
print(f"\nGroup mapping:")
for kv_idx in range(KV):
    q_idx = list(range(kv_idx * groups, (kv_idx + 1) * groups))
    print(f"  KV[{kv_idx}] → Q{q_idx}")


## 4. SwiGLU Intermediate Width

A basic Transformer FFN expands from $d$ to $4d$ and projects back, suggesting 2304 for `hidden_size=576`. The configuration instead uses `intermediate_size=1536` because a **SwiGLU FFN** has gate, up, and down matrices rather than only up and down.

Using $4d$ with three matrices would increase parameters substantially. SmolLM2 chooses a narrower intermediate size while retaining 576-dimensional input and output.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

import torch

class LlamaFFN(nn.Module):
    """Llama-style SwiGLU FFN: gate does gating, up does projection, down projects back"""
    def __init__(self, dim, intermediate_dim):
        super().__init__()
        self.gate = nn.Linear(dim, intermediate_dim, bias=False)
        self.up   = nn.Linear(dim, intermediate_dim, bias=False)
        self.down = nn.Linear(intermediate_dim, dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

ffn = LlamaFFN(D, FF)
x = torch.randn(2, 16, D)
out = ffn(x)

gate_p = sum(p.numel() for p in ffn.gate.parameters())
up_p   = sum(p.numel() for p in ffn.up.parameters())
down_p = sum(p.numel() for p in ffn.down.parameters())

print(f"LlamaFFN structure:")
print(f"  gate: {list(ffn.gate.weight.shape)}  ({gate_p:,} params)")
print(f"  up:   {list(ffn.up.weight.shape)}  ({up_p:,} params)")
print(f"  down: {list(ffn.down.weight.shape)}  ({down_p:,} params)")
print(f"  Total: {gate_p + up_p + down_p:,} params")
print(f"\nForward: input [2, 16, {D}], intermediate [2, 16, {FF}], output [2, 16, {D}]")
print(f"Input shape: {list(x.shape)}  Output shape: {list(out.shape)}")
print("Key observation: SwiGLU uses three matrices: gate, up, and down.")


## 5. Configuration and Weight Compatibility

`rms_norm_eps=1e-5` does not change a tensor width but still changes computation. **RMSNorm** scales a vector by its root mean square without first subtracting its mean.

LayerNorm commonly has weight and bias; this RMSNorm has weight only. Reconstructing it as LayerNorm would create bias parameters absent from the checkpoint. We compare both on one input.


In [ ]:
# Using PyTorch's built-in RMSNorm and LayerNorm directly
import torch
import torch.nn as nn

rn = nn.RMSNorm(D, eps=config['rms_norm_eps'])
ln = nn.LayerNorm(D, eps=config['rms_norm_eps'])

print("=== nn.RMSNorm structure ===")
print(rn)
print("\n=== nn.LayerNorm structure ===")
print(ln)

# Same random input
x = torch.randn(4, 8, D)
with torch.no_grad():
    ln_out = ln(x)
    rn_out = rn(x)

print(f"\nLayerNorm:  weight + bias = {sum(p.numel() for p in ln.parameters())} params")
print(f"RMSNorm:   weight only    = {sum(p.numel() for p in rn.parameters())} params")
print(f"\nBefore normalization:  mean={x.mean():.3f}, std={x.std():.3f}")
print(f"LayerNorm: mean={ln_out.mean():.6f}, std={ln_out.std():.3f}  <- mean is 0")
print(f"RMSNorm:   mean={rn_out.mean():.4f}, std={rn_out.std():.3f}  <- mean is NOT 0")
print(f"\nSaves {D} bias params per layer, {D * 30:,} total across 30 layers")


## 6. Parameter Counting

Now calculate every major parameter group by formula and compare it with PyTorch's `sum(p.numel())`. A discrepancy means at least one configuration field was mapped incorrectly.

`tie_word_embeddings=True` matters here: the output layer reuses the input Embedding instead of adding another `[49152, 576]` matrix.


In [ ]:
# ========== Theoretical calculation: item by item using formulas ==========
# Per-layer Attention: Q, K, V, O four projections
import torch.nn as nn
import torch.nn.functional as F

q_p  = D * H * head_dim       # Q: [D, H*head_dim]
k_p  = D * KV * head_dim      # K: [D, KV*head_dim]
v_p  = D * KV * head_dim      # V: [D, KV*head_dim]
o_p  = H * head_dim * D       # O: [H*head_dim, D]
attn_p = q_p + k_p + v_p + o_p

# Per-layer FFN: gate, up, down three projections
ffn_p = 3 * D * FF

# Per-layer Norm: 2 RMSNorm, each with only D weights
norm_p = 2 * D

per_layer = attn_p + ffn_p + norm_p
total_theory = V * D + L * per_layer + D  # Embedding + 30 layers + final Norm

print("========== Theoretical Calculation (formulas) ==========")
print(f"Per-layer Attention (Q+K+V+O): {attn_p:>10,}")
print(f"Per-layer FFN (gate+up+down):   {ffn_p:>10,}")
print(f"Per-layer RMSNorm x 2:          {norm_p:>10,}")
print(f"Per-layer subtotal:             {per_layer:>10,} ~ {per_layer/1e6:.2f}M")
print(f"\nEmbedding ({V}×{D}):       {V*D:>10,} ≈ {V*D/1e6:.1f}M")
print(f"{L} layers of Blocks:        {L*per_layer:>10,} ~ {L*per_layer/1e6:.1f}M")
print(f"Final RMSNorm:                  {D:>10,}")
print(f"Theoretical total:              {total_theory:>10,} ~ {total_theory/1e6:.1f}M")

# ========== Actual calculation: build the full model, use sum(p.numel()) ==========
class SmolLM2ConfigModel(nn.Module):
    """Assemble SmolLM2 from config: Embedding -> 30x Block -> RMSNorm -> output projection"""
    def __init__(self, config):
        super().__init__()
        d, V = config['hidden_size'], config['vocab_size']
        self.embed = nn.Embedding(V, d)
        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config['num_hidden_layers'])])
        self.final_norm = nn.RMSNorm(d, eps=config['rms_norm_eps'])

    def forward(self, x):
        x = self.embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.final_norm(x)
        # tie_word_embeddings=True: output projection reuses embed.weight, no separate lm_head
        return F.linear(x, self.embed.weight)

model = SmolLM2ConfigModel(config)
total_real = sum(p.numel() for p in model.parameters())

print("\n========== Actual Calculation (sum(p.numel())) ==========")
emb_real = sum(p.numel() for p in model.embed.parameters())
blocks_real = sum(p.numel() for p in model.blocks.parameters())
norm_real = sum(p.numel() for p in model.final_norm.parameters())
print(f"Embedding:     {emb_real:>10,}  ({V}×{D})")
print(f"30×Block:      {blocks_real:>10,}  ({L}×{per_layer:,})")
print(f"final_norm:    {norm_real:>10,}  ({D})")
print(f"{'─'*45}")
print(f"Actual total:  {total_real:>10,} ~ {total_real/1e6:.1f}M")

print(f"\n========== Comparison ==========")
print(f"Theoretical: {total_theory:,}")
print(f"Actual:      {total_real:,}")
print(f"Match:       {total_theory == total_real}")
if total_theory == total_real:
    print("The theoretical formulas match PyTorch's actual parameters exactly")
else:
    print(f"Difference of {abs(total_theory - total_real):,}, please check")


Once totals agree, ask where parameters are allocated. The chart shows that RMSNorm contributes very little; the 30 FFNs and Embedding dominate.


In [ ]:
import matplotlib.pyplot as plt

emb_s = V * D
attn_s = attn_p * L
ffn_s = ffn_p * L
norm_s = norm_p * L

labels = [f'Embedding\n{emb_s/1e6:.1f}M', f'Attention×{L}\n{attn_s/1e6:.1f}M',
          f'FFN×{L}\n{ffn_s/1e6:.1f}M', f'RMSNorm×{L}\n{norm_s/1e6:.2f}M']
sizes = [emb_s, attn_s, ffn_s, norm_s]

plt.figure(figsize=(6, 5))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90,
        colors=['#5DADE2','#F5B041','#E74C3C','#58D68D'])
plt.title(f'SmolLM2-135M parameter distribution')
plt.tight_layout()
plt.show()

## 7. Three Kinds of Configuration

`config.json` describes internal model structure, while tokenization depends on vocabulary, merge rules, and special-token configuration.

The real SmolLM2 `tokenizer_config.json` also demonstrates that absent fields must not be invented. It assigns `<|endoftext|>` as BOS, EOS, and unknown token, but specifies no pad token and no `add_bos_token=True` switch.


In [ ]:
tokenizer_config = {
    "bos_token": "<|endoftext|>",
    "eos_token": "<|endoftext|>",
    "unk_token": "<|endoftext|>",
    "model_max_length": 8192,
    "tokenizer_class": "GPT2Tokenizer",
    "vocab_size": 49152,
}

print("tokenizer_config.json (simplified):")
print(json.dumps(tokenizer_config, indent=2, ensure_ascii=False))


The model–Tokenizer interface has one hard constraint: every emitted token ID must be a valid row of the Embedding table. In the real vocabulary, `<|endoftext|>` is ID 0, `<|im_start|>` is 1, and `<|im_end|>` is 2. We check the boundary with a short ID sequence.


In [ ]:
special_token_ids = {
    "<|endoftext|>": 0,
    "<|im_start|>": 1,
    "<|im_end|>": 2,
}
token_ids = [0, 1847, 302, 2]

print("Special Token to ID mapping:", special_token_ids)
print('Result:', token_ids)
print(f"Valid ID range for Embedding: 0 through {V - 1}")
assert min(token_ids) >= 0 and max(token_ids) < V
print("Key observation: every ID addresses one row of the Embedding table.")


In [ ]:
required_behavior_fields = ["pad_token", "padding_side", "add_bos_token"]
missing_fields = [key for key in required_behavior_fields if key not in tokenizer_config]

print("Not found in this file:", missing_fields)
print("Key observation: a missing field is neither True nor an assumed conventional value.")
print("If the calling code needs padding, it must specify a strategy or use the library default explicitly.")


### 7.1 Default Configuration and Request Parameters

After producing logits, generation must select the next token. This policy does not alter weight shapes.

SmolLM2-135M's actual `generation_config.json` contains BOS, EOS, and a Transformers version, but no defaults such as `temperature=0.6`, `top_p=0.9`, or `top_k=50`. A request may supply these values, but they must not be presented as repository defaults.


In [ ]:
generation_config = {
    "_from_model_config": True,
    "bos_token_id": 0,
    "eos_token_id": 0,
    "transformers_version": "4.40.1",
}

request_config = {
    "temperature": 0.6,
    "top_p": 0.9,
    "top_k": 50,
    "max_new_tokens": 256,
    "do_sample": True,
}

print("generation_config.json in the repository:")
print(json.dumps(generation_config, indent=2, ensure_ascii=False))


#### Optional: Request Parameters and Candidate Distributions

The following `request_config` is defined for this experiment rather than read from SmolLM2. Structural configuration determines how logits are computed; request configuration determines how a token is selected.

Temperature scales logits before softmax. $T<1$ sharpens the distribution; $T>1$ flattens it. The generation chapter implements full sampling, while this section only visualizes the probabilities.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import torch

tokens = ["the", "to", "and", "of", "in", "is", "you", "that", "it", "for"]
logits = torch.tensor([3.2, 2.8, 2.5, 2.0, 1.5, 1.2, 0.9, 0.6, 0.3, 0.1])

temperatures = [0.3, 0.6, 1.0, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for ax, T in zip(axes, temperatures):
    probs = torch.softmax(logits / T, dim=-1).numpy()
    colors = ['#E74C3C' if p == probs.max() else '#BDC3C7' for p in probs]
    ax.bar(range(len(tokens)), probs, color=colors)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"T={T}")
    ax.set_ylabel("Probability")
    # Annotate the top 2 probabilities
    top2 = np.argsort(probs)[-2:]
    ax.annotate(f"{probs[top2[0]]:.2f}", (top2[0], probs[top2[0]]),
                ha='center', va='bottom', fontsize=9)
    ax.annotate(f"{probs[top2[1]]:.2f}", (top2[1], probs[top2[1]]),
                ha='center', va='bottom', fontsize=9)

plt.suptitle("Effect of temperature on probability distribution", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Key observation: at T=0.3 the highest-probability token dominates; at T=2.0 probabilities become flatter.")
request_temperature = request_config["temperature"]
print(f"This request uses T={request_temperature}; it is not the model repository's default.")


What does `top_p` retain?

Sort tokens by decreasing probability and keep the smallest set whose cumulative probability reaches $p$. This is nucleus sampling. Concentrated distributions retain fewer candidates; flatter distributions retain more.


In [ ]:
import torch
import matplotlib.pyplot as plt

probs = torch.softmax(logits, dim=-1)
sorted_probs, sorted_indices = torch.sort(probs, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left plot: cumulative probability
ax1.bar(range(len(tokens)), sorted_probs.numpy(), color='#3498DB', alpha=0.7)
ax1.plot(range(len(tokens)), cumsum.numpy(), 'o-', color='#E74C3C', linewidth=2, markersize=6)
ax1.axhline(y=0.9, color='gray', linestyle='--', alpha=0.7, label='top_p=0.9')
ax1.set_xticks(range(len(tokens)))
ax1.set_xticklabels([tokens[i] for i in sorted_indices])
ax1.set_ylabel("Probability / cumulative probability")
ax1.legend()

# Right plot: tokens kept after top_p=0.9
mask = cumsum <= 0.9
keep_n = mask.sum().item() + 1  # +1 because we need to include the first token that exceeds the threshold
keep_indices = sorted_indices[:keep_n]
keep_probs = probs[keep_indices]
filtered = torch.zeros_like(probs)
filtered[keep_indices] = keep_probs / keep_probs.sum()

ax2.bar(range(len(tokens)), filtered.numpy(),
        color=['#27AE60' if i in keep_indices else '#BDC3C7' for i in range(len(tokens))])
ax2.set_xticks(range(len(tokens)))
ax2.set_xticklabels(tokens)
ax2.set_ylabel("Renormalized probability")
ax2.set_title(f"top_p=0.9: keep {keep_n} tokens, drop the rest")

plt.tight_layout()
plt.show()

kept_tokens = [tokens[i] for i in sorted_indices[:keep_n]]
print(f"top_p=0.9 retains the first {keep_n} Tokens: {kept_tokens}")
print(f"Discarded tokens: {[tokens[i] for i in sorted_indices[keep_n:]]}")


How does `top_k` differ from `top_p`?

`top_k` retains a fixed number of highest-probability tokens. It controls candidate count, while `top_p` controls covered probability mass. The toy vocabulary below shows the distinction.


In [ ]:
import torch
import matplotlib.pyplot as plt

K_list = [3, 5, 10, len(tokens)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for ax, K in zip(axes, K_list):
    topk_indices = sorted_indices[:K]
    topk_probs = probs[topk_indices]
    topk_probs = topk_probs / topk_probs.sum()
    display = torch.zeros_like(probs)
    display[topk_indices] = topk_probs
    colors = ['#27AE60' if i in topk_indices else '#BDC3C7' for i in range(len(tokens))]
    ax.bar(range(len(tokens)), display.numpy(), color=colors)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"top_k={K}")

plt.suptitle("Effect of top_k truncation on candidate tokens", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

request_top_k = request_config["top_k"]
print(f"One request sets top_k={request_top_k}.")
print(f"From {V:,} Tokens, it retains the top {request_top_k}.")
print(f"The candidates account for {request_top_k / V * 100:.2f}% of the vocabulary.")


Combining Three Request Parameters

Temperature, `top_k`, and `top_p` may act together. The teaching function scales logits, constructs both filters, and takes their intersection. Framework ordering and boundary behavior may differ, so production behavior must be verified against the relevant version.


In [ ]:
import torch

def sample_filter(all_tokens, logits, T=1.0, top_k=None, top_p=None):
    """Apply temperature, top_k, top_p and return the list of kept tokens"""
    scaled = logits / T
    probs = torch.softmax(scaled, dim=-1)
    allowed = torch.ones_like(probs, dtype=torch.bool)

    # top_k cannot exceed vocabulary size
    k = min(top_k, len(all_tokens)) if top_k is not None else None
    if k is not None:
        topk_vals, _ = torch.topk(probs, k)
        threshold = topk_vals[-1]
        allowed = allowed & (probs >= threshold)

    if top_p is not None:
        sorted_p, sorted_i = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_p, dim=-1)
        n_keep = min((cum <= top_p).sum().item() + 1, len(all_tokens))
        keep_i = sorted_i[:n_keep]
        mask = torch.zeros_like(probs, dtype=torch.bool)
        mask[keep_i] = True
        allowed = allowed & mask

    return [all_tokens[i] for i in range(len(all_tokens)) if allowed[i]]

# Three configurations (top_k adjusted to fit the demo vocabulary size)
configs = [
    ("Conservative", 0.3, 3, 0.85),
    ("Greedy (T->0)", 0.001, None, None),
    ("Creative", 1.2, None, 0.95),
]

logits_np = logits.clone()
print(f"{'Config':<20} {'Kept tokens'}")
print("-" * 50)
for name, T, tk, tp in configs:
    kept = sample_filter(tokens, logits_np, T=T, top_k=tk, top_p=tp)
    print(f"{name:<20} {kept}")

print("\nKey observation: in this teaching implementation, the final candidates are the intersection of both filters.")
print("Conservative settings retain fewer candidates; higher temperature and top_p retain more.")


## 8. Four Stages of an Inference Request

The final diagram connects the full path. Its fourth stage says “repository defaults plus request parameters” because caller-provided values may override defaults; a repository file alone cannot prove which sampling settings a request used.


In [ ]:
stages = [
    ("1. Text → Token IDs", "Tokenizer files", "vocabulary and special tokens"),
    ("2. Token IDs → model", "config.json", "shapes and architecture"),
    ("3. Model → logits", "model weights", "learned parameter values"),
    ("4. Logits → next Token", "generation defaults + request", "sampling policy"),
]

for stage, source, role in stages:
    print(f"{stage:<25} | {source:<29} | {role}")

print("\nKey observation: each of the four stages can change independently, but their interfaces must stay compatible.")
print("For example, Token IDs must be below vocab_size, and parameter shapes must match the weight file.")


## Summary

- [ ] A HuggingFace model repository has 3 core JSON configs: config.json, tokenizer_config.json, generation_config.json
- [ ] Every number in config.json corresponds to a concrete PyTorch module: Embedding, Q/K/V/O, FFN, RMSNorm
- [ ] hidden_size corresponds to d_model, and intermediate_size corresponds to the FFN's d_ff
- [ ] GQA saves parameters and memory by reducing the number of KV heads: K projection parameters are only 1/3 of Q
- [ ] Llama's FFN has an additional gate matrix compared to the teaching version (three weights vs two weights), using SiLU for gating
- [ ] RMSNorm does not perform mean centering, and usually lacks LayerNorm's beta shift parameter; it mainly changes the normalization method, and saving parameters is just an incidental difference
- [ ] tokenizer_config.json controls BOS/EOS insertion, padding direction, and maximum length
- [ ] The three core parameters in generation_config.json (temperature, top_p, top_k) jointly determine sampling behavior
- [ ] Changing any one of these config files changes the model's behavior

Once you learn to read these config files, you essentially have a complete specification for the model. Open any new model on HuggingFace: first check config.json for structural dimensions, then tokenizer_config.json for special tokens, and finally generation_config.json for sampling strategy -- after reading all three, the model's design philosophy becomes clear.


## Exercises

> You can ask AI to help explain the approach, but it's not recommended to let AI "do this problem for you" directly.

**Exercise 1: Recover GQA Grouping from Configuration**

Given `hidden_size=576`, `num_attention_heads=9`, and `num_key_value_heads=3`, calculate head dimension and the number of Q heads served by each K/V head.

Hint: perform two integer divisions and check for remainders.


In [ ]:
# TODO: replace None with the calculation
exercise_head_dim = None
q_heads_per_kv_group = None

assert exercise_head_dim == 64, "Recalculate 576 / 9"
assert q_heads_per_kv_group == 3, "Recalculate 9 / 3"
print("Exercise 1 passed: you can recover the basic GQA shapes from a configuration.")


**Exercise 2: Change Configuration While Preserving Shape Constraints**

Set `hidden_size=768`, with 12 Q heads and four K/V heads, then complete the configuration and parameter-count check.

Hint: `head_dim` remains 64 and each K/V head still serves three Q heads.


In [ ]:
# Exercise 2: complete the three TODOs, then observe the parameter-count change
my_config = {
    "vocab_size": 49152,
    "hidden_size": None,          # TODO: 768
    "num_hidden_layers": 30,
    "num_attention_heads": None, # TODO: 12
    "num_key_value_heads": None, # TODO: 4
    "intermediate_size": 2048,
}

V2, D2, L2, H2, KV2, FF2 = (my_config[k] for k in (
    "vocab_size", "hidden_size", "num_hidden_layers",
    "num_attention_heads", "num_key_value_heads", "intermediate_size"))
assert D2 == 768, "Enter hidden_size"
assert H2 == 12, "Enter num_attention_heads"
assert KV2 == 4, "Enter num_key_value_heads"
assert D2 % H2 == 0 and H2 % KV2 == 0, "Head counts and dimensions must satisfy the divisibility constraints"
hd2 = D2 // H2

emb_p = V2 * D2
attn_p = D2 * H2 * hd2 + D2 * KV2 * hd2 * 2 + H2 * hd2 * D2
ffn_p = 3 * D2 * FF2
per_layer_2 = attn_p + ffn_p + 2 * D2
total_p = emb_p + L2 * per_layer_2 + D2

assert hd2 == 64 and H2 // KV2 == 3
print(f"Exercise 2 passed: the new configuration has about {total_p / 1e6:.1f}M parameters.")
print(f"Embedding has {emb_p:,} parameters, and each Block has {per_layer_2:,} parameters.")


**Exercise 3: Estimate LLaMA-7B Parameter Count**

In the approximation, $V$ is vocabulary size, $d$ hidden size, and $L$ layer count. The term $12d^2$ approximates Attention and FFN parameters per layer; $2Vd$ counts separate input Embedding and untied output weights.

Hint: calculate two $V\times d$ matrices and 32 Blocks. Norms and other small terms are omitted, so the result should be close rather than exact.


In [ ]:
# Exercise 3: Verify the parameter count estimation formula
d = 4096
L = 32
vocab_size = 32000

# TODO: how many parameters do input Embedding and the output layer share?
embedding_and_head_params = None

# TODO: Per-Block parameter count (using the empirical formula 12 * d^2)
block_params = None

# TODO: Total parameter count = Embedding + L layers of Block
total_params = None

# Verify
assert embedding_and_head_params is not None, 'Please replace the placeholder before running the assertion.'
assert block_params is not None, 'Please replace the placeholder before running the assertion.'
assert total_params is not None, 'Please replace the placeholder before running the assertion.'

expected_emb_and_head = 2 * vocab_size * d
expected_block = 12 * d * d
expected_total = expected_emb_and_head + L * expected_block

assert embedding_and_head_params == expected_emb_and_head
assert block_params == expected_block
assert total_params == expected_total

print("Exercise 3 passed: you can use a configuration for order-of-magnitude estimates.")
print(f"Input Embedding + output layer: {embedding_and_head_params / 1e9:.2f}B")
print(f"{L} Blocks: {L * block_params / 1e9:.2f}B")
print(f"   Total params: {total_params/1e9:.2f}B")


## References

- [SmolLM2-135M model repository](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/tree/main) -- source of all JSON files analyzed in this section
- Touvron et al., [LLaMA: Open and Efficient Foundation Language Models](https://arxiv.org/abs/2302.13971), 2023
- Su et al., [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864), 2021
- Ainslie et al., [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245), 2023
- Holtzman et al., [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751), 2020 -- original paper on top_p (nucleus sampling)
